# STEMNIST 原始压力帧准备

本 Notebook 完成以下工作：

1. 从原始 ZIP 中读取 `RawCharacters/*.h5`。
2. 每个样本只读取 `pressure_data`，保持原始 `uint8 [240, 16, 16]`。
3. 按类别精确划分为 `train:val:test = 70:15:15`。
4. 使用种子 `42` 和 SHA-256 确定性排序，保证相同 ZIP 在不同设备上得到相同划分。
5. 每个集合输出一个 `pressure.npy` 和一个 `manifest.csv`。

最终目录：

```text
data/pressure/
├── metadata.json
├── split_manifest.csv
├── train/
│   ├── pressure.npy
│   └── manifest.csv
├── val/
│   ├── pressure.npy
│   └── manifest.csv
└── test/
    ├── pressure.npy
    └── manifest.csv
```

> 这是样本级类别分层划分，同一参与者可能同时出现在三个集合中。

In [1]:
# 1. 导入依赖并设置路径
from __future__ import annotations

import csv
import io
import json
import os
import shutil
import tempfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from hashlib import md5, sha256
from pathlib import Path, PurePosixPath
from zipfile import ZipFile, ZipInfo

import h5py
import numpy as np


# 如果原始 ZIP 不在 data 目录，只需要修改 ZIP_PATH。
def find_project_root(start: Path | None = None) -> Path:
    """Find the project root portably on Windows and Linux."""
    override = os.getenv("STEMNIST_PROJECT_ROOT")
    if override:
        project_root = Path(override).expanduser().resolve()
        if not project_root.is_dir():
            raise NotADirectoryError(
                f"STEMNIST_PROJECT_ROOT is not a directory: {project_root}"
            )
        return project_root

    current = (start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "STEMNIST Dataset.zip").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. Run the notebook from inside the project "
        "or set the STEMNIST_PROJECT_ROOT environment variable."
    )


PROJECT_ROOT = find_project_root()
ZIP_PATH = PROJECT_ROOT / "data" / "STEMNIST Dataset.zip"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "pressure"

EXPECTED_ZIP_MD5 = "6ca4638b2f95bf34f59873ab62399bd8"
EXPECTED_SHAPE = (240, 16, 16)
EXPECTED_DTYPE = np.dtype(np.uint8)

LABELS = tuple("ABCDEFGHIJKLMNOPQRSTUVWXYZ123456789")
LABEL_TO_INDEX = {label: index for index, label in enumerate(LABELS)}

SEED = 42
SPLIT_COUNTS_PER_CLASS = {"train": 154, "val": 33, "test": 33}
EXPECTED_CLASS_SIZE = sum(SPLIT_COUNTS_PER_CLASS.values())  # 220
EXPECTED_SAMPLE_COUNT = EXPECTED_CLASS_SIZE * len(LABELS)  # 7700

# 默认不覆盖已有结果；确认需要重建时改成 True。
OVERWRITE = False

In [2]:
# 2. 扫描 ZIP，并读取单个压力样本
@dataclass(frozen=True)
class SampleRecord:
    member_name: str
    sample_id: str
    participant_id: str
    label: str
    label_index: int
    repetition: int


def calculate_md5(path: Path, block_size: int = 1024 * 1024) -> str:
    """分块计算文件 MD5，避免一次读入整个 ZIP。"""
    digest = md5()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def is_pressure_member(info: ZipInfo) -> bool:
    """只接受 RawCharacters 目录中的 .h5 文件。"""
    if info.is_dir():
        return False

    path = PurePosixPath(info.filename.replace("\\", "/"))
    return (
        path.suffix.lower() == ".h5"
        and len(path.parts) >= 2
        and path.parts[-2] == "RawCharacters"
    )


def parse_record(member_name: str) -> SampleRecord:
    """从 AT_A_1.h5 解析参与者、类别和重复次数。"""
    sample_id = PurePosixPath(member_name.replace("\\", "/")).stem
    parts = sample_id.split("_")
    if len(parts) != 3:
        raise RuntimeError(f"无法解析样本文件名：{member_name}")

    participant_id, label, repetition_text = parts
    if label not in LABEL_TO_INDEX:
        raise RuntimeError(f"出现未知标签 {label!r}：{member_name}")

    try:
        repetition = int(repetition_text)
    except ValueError as error:
        raise RuntimeError(f"重复次数不是整数：{member_name}") from error

    return SampleRecord(
        member_name=member_name,
        sample_id=sample_id,
        participant_id=participant_id,
        label=label,
        label_index=LABEL_TO_INDEX[label],
        repetition=repetition,
    )


def scan_records(archive: ZipFile) -> list[SampleRecord]:
    """按成员名称排序后扫描，避免 ZIP 内部顺序影响结果。"""
    member_names = sorted(
        info.filename
        for info in archive.infolist()
        if is_pressure_member(info)
    )

    if len(member_names) != EXPECTED_SAMPLE_COUNT:
        raise RuntimeError(
            f"应找到 {EXPECTED_SAMPLE_COUNT} 个压力样本，"
            f"实际找到 {len(member_names)} 个"
        )
    if len(member_names) != len(set(member_names)):
        raise RuntimeError("ZIP 中存在重复成员名称")

    records = [parse_record(name) for name in member_names]
    sample_ids = [record.sample_id for record in records]
    if len(sample_ids) != len(set(sample_ids)):
        raise RuntimeError("ZIP 中存在重复 sample_id")

    class_counts = Counter(record.label for record in records)
    invalid_counts = {
        label: class_counts.get(label, 0)
        for label in LABELS
        if class_counts.get(label, 0) != EXPECTED_CLASS_SIZE
    }
    if invalid_counts:
        raise RuntimeError(f"类别样本数异常：{invalid_counts}")

    return records


def read_pressure(archive: ZipFile, record: SampleRecord) -> np.ndarray:
    """从一个 HDF5 成员中只读取 pressure_data。"""
    member_bytes = archive.read(record.member_name)

    try:
        with h5py.File(io.BytesIO(member_bytes), "r") as h5_file:
            if "pressure_data" not in h5_file:
                raise RuntimeError(
                    f"{record.member_name} 缺少 pressure_data"
                )

            dataset = h5_file["pressure_data"]
            if tuple(dataset.shape) != EXPECTED_SHAPE:
                raise RuntimeError(
                    f"{record.member_name} 形状错误：{dataset.shape}"
                )
            if np.dtype(dataset.dtype) != EXPECTED_DTYPE:
                raise RuntimeError(
                    f"{record.member_name} 类型错误：{dataset.dtype}"
                )

            return np.asarray(dataset[...], dtype=np.uint8)
    except OSError as error:
        raise RuntimeError(
            f"无法读取 HDF5：{record.member_name}"
        ) from error

In [3]:
# 3. 使用种子参与 SHA-256 排序，完成跨设备一致的分层划分
def deterministic_key(
    sample_id: str,
    *,
    seed: int,
    namespace: str,
) -> bytes:
    """生成与设备和 Python 随机数实现无关的排序键。"""
    message = f"stemnist-v1\0{namespace}\0{seed}\0{sample_id}"
    return sha256(message.encode("utf-8")).digest()


def stratified_split(
    records: list[SampleRecord],
    seed: int = SEED,
) -> dict[str, list[SampleRecord]]:
    """每类固定分为 154/33/33，得到精确的 70/15/15。"""
    records_by_label: dict[str, list[SampleRecord]] = defaultdict(list)
    for record in records:
        records_by_label[record.label].append(record)

    splits: dict[str, list[SampleRecord]] = {
        "train": [],
        "val": [],
        "test": [],
    }

    for label in LABELS:
        class_records = records_by_label[label]
        if len(class_records) != EXPECTED_CLASS_SIZE:
            raise RuntimeError(
                f"类别 {label} 应有 {EXPECTED_CLASS_SIZE} 个样本，"
                f"实际为 {len(class_records)}"
            )

        ordered = sorted(
            class_records,
            key=lambda record: (
                deterministic_key(
                    record.sample_id,
                    seed=seed,
                    namespace="membership",
                ),
                record.sample_id,
            ),
        )

        train_end = SPLIT_COUNTS_PER_CLASS["train"]
        val_end = train_end + SPLIT_COUNTS_PER_CLASS["val"]
        splits["train"].extend(ordered[:train_end])
        splits["val"].extend(ordered[train_end:val_end])
        splits["test"].extend(ordered[val_end:])

    # 再做一次确定性全局排序，避免输出数组按类别成块排列。
    for split_name, split_records in splits.items():
        splits[split_name] = sorted(
            split_records,
            key=lambda record: (
                deterministic_key(
                    record.sample_id,
                    seed=seed,
                    namespace=f"order:{split_name}",
                ),
                record.sample_id,
            ),
        )

    all_ids = [
        record.sample_id
        for split_records in splits.values()
        for record in split_records
    ]
    if len(all_ids) != EXPECTED_SAMPLE_COUNT:
        raise RuntimeError("划分后的总样本数不正确")
    if len(all_ids) != len(set(all_ids)):
        raise RuntimeError("三个集合之间存在重复样本")

    expected_sizes = {
        name: count * len(LABELS)
        for name, count in SPLIT_COUNTS_PER_CLASS.items()
    }
    actual_sizes = {name: len(value) for name, value in splits.items()}
    if actual_sizes != expected_sizes:
        raise RuntimeError(
            f"划分大小错误：期望 {expected_sizes}，实际 {actual_sizes}"
        )

    return splits

In [4]:
# 4. 将每个集合写成一个 pressure.npy，并保存样本清单
MANIFEST_FIELDS = (
    "row_index",
    "sample_id",
    "participant_id",
    "label",
    "label_index",
    "repetition",
    "source_member",
    "split",
    "split_key",
)


def manifest_row(
    record: SampleRecord,
    *,
    row_index: int,
    split_name: str,
    seed: int,
) -> dict[str, object]:
    return {
        "row_index": row_index,
        "sample_id": record.sample_id,
        "participant_id": record.participant_id,
        "label": record.label,
        "label_index": record.label_index,
        "repetition": record.repetition,
        "source_member": record.member_name,
        "split": split_name,
        "split_key": deterministic_key(
            record.sample_id,
            seed=seed,
            namespace="membership",
        ).hex(),
    }


def close_memmap(array: np.memmap) -> None:
    """刷新并关闭映射，确保 Windows 可以立即移动文件。"""
    array.flush()
    memory_map = getattr(array, "_mmap", None)
    if memory_map is not None:
        memory_map.close()


def export_one_split(
    archive: ZipFile,
    split_name: str,
    records: list[SampleRecord],
    split_dir: Path,
    seed: int,
) -> list[dict[str, object]]:
    """导出一个集合，内存中始终只保留一个原始样本。"""
    split_dir.mkdir(parents=True, exist_ok=False)
    pressure_path = split_dir / "pressure.npy"
    manifest_path = split_dir / "manifest.csv"

    pressure_array = np.lib.format.open_memmap(
        pressure_path,
        mode="w+",
        dtype=EXPECTED_DTYPE,
        shape=(len(records), *EXPECTED_SHAPE),
    )
    rows: list[dict[str, object]] = []

    try:
        with manifest_path.open("w", encoding="utf-8", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=MANIFEST_FIELDS)
            writer.writeheader()

            for row_index, record in enumerate(records):
                pressure_array[row_index] = read_pressure(archive, record)
                row = manifest_row(
                    record,
                    row_index=row_index,
                    split_name=split_name,
                    seed=seed,
                )
                writer.writerow(row)
                rows.append(row)

                completed = row_index + 1
                if completed % 500 == 0 or completed == len(records):
                    print(
                        f"{split_name}: 已处理 "
                        f"{completed}/{len(records)}"
                    )
    finally:
        close_memmap(pressure_array)

    return rows


def validate_output_target(output_root: Path, project_root: Path) -> None:
    """防止覆盖项目目录或项目以外的路径。"""
    output_root = output_root.resolve()
    project_root = project_root.resolve()
    if output_root == project_root or project_root not in output_root.parents:
        raise ValueError(
            f"输出目录必须位于项目目录内部：{output_root}"
        )


def prepare_pressure_data(
    zip_path: Path = ZIP_PATH,
    output_root: Path = OUTPUT_ROOT,
    *,
    seed: int = SEED,
    overwrite: bool = OVERWRITE,
) -> dict[str, object]:
    """校验 ZIP、确定性划分并生成三个压力帧数组。"""
    zip_path = Path(zip_path).expanduser().resolve()
    output_root = Path(output_root).expanduser().resolve()
    project_root = PROJECT_ROOT.expanduser().resolve()

    if not zip_path.is_file():
        raise FileNotFoundError(f"找不到原始 ZIP：{zip_path}")
    validate_output_target(output_root, project_root)

    actual_md5 = calculate_md5(zip_path)
    if actual_md5.lower() != EXPECTED_ZIP_MD5.lower():
        raise RuntimeError(
            "ZIP MD5 校验失败：\n"
            f"期望：{EXPECTED_ZIP_MD5}\n"
            f"实际：{actual_md5}"
        )

    output_has_files = (
        output_root.exists() and any(output_root.iterdir())
    )
    if output_has_files and not overwrite:
        raise FileExistsError(
            f"输出目录已经存在：{output_root}\n"
            "确认需要重建时，将 OVERWRITE 改为 True。"
        )

    output_root.parent.mkdir(parents=True, exist_ok=True)
    staging_root = Path(
        tempfile.mkdtemp(
            prefix=".pressure-building-",
            dir=output_root.parent,
        )
    )

    try:
        with ZipFile(zip_path, "r") as archive:
            print("正在扫描 ZIP……")
            records = scan_records(archive)
            splits = stratified_split(records, seed=seed)

            all_rows: list[dict[str, object]] = []
            for split_name in ("train", "val", "test"):
                rows = export_one_split(
                    archive,
                    split_name,
                    splits[split_name],
                    staging_root / split_name,
                    seed,
                )
                all_rows.extend(rows)

        with (staging_root / "split_manifest.csv").open(
            "w",
            encoding="utf-8",
            newline="",
        ) as file:
            writer = csv.DictWriter(file, fieldnames=MANIFEST_FIELDS)
            writer.writeheader()
            writer.writerows(all_rows)

        metadata = {
            "source_zip": str(zip_path),
            "source_zip_md5": actual_md5,
            "split_seed": seed,
            "split_algorithm": "per-class-sha256-sort-v1",
            "ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
            "counts": {
                name: len(split_records)
                for name, split_records in splits.items()
            },
            "pressure_shape": list(EXPECTED_SHAPE),
            "pressure_dtype": str(EXPECTED_DTYPE),
            "labels": list(LABELS),
        }
        with (staging_root / "metadata.json").open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(metadata, file, ensure_ascii=False, indent=2)

        if output_root.exists():
            shutil.rmtree(output_root)
        staging_root.replace(output_root)

    except Exception:
        shutil.rmtree(staging_root, ignore_errors=True)
        raise

    print(f"完成：{output_root}")
    return metadata

In [5]:
# 5. 执行数据准备
# 首次运行保持 OVERWRITE=False；需要重建时再改为 True。
metadata = prepare_pressure_data(
    zip_path=ZIP_PATH,
    output_root=OUTPUT_ROOT,
    seed=SEED,
    overwrite=OVERWRITE,
)
metadata

正在扫描 ZIP……
train: 已处理 500/5390
train: 已处理 1000/5390
train: 已处理 1500/5390
train: 已处理 2000/5390
train: 已处理 2500/5390
train: 已处理 3000/5390
train: 已处理 3500/5390
train: 已处理 4000/5390
train: 已处理 4500/5390
train: 已处理 5000/5390
train: 已处理 5390/5390
val: 已处理 500/1155
val: 已处理 1000/1155
val: 已处理 1155/1155
test: 已处理 500/1155
test: 已处理 1000/1155
test: 已处理 1155/1155
完成：C:\Users\Fortyfour\Desktop\STEMNIST_Classify\data\pressure


{'source_zip': 'C:\\Users\\Fortyfour\\Desktop\\STEMNIST_Classify\\data\\STEMNIST Dataset.zip',
 'source_zip_md5': '6ca4638b2f95bf34f59873ab62399bd8',
 'split_seed': 42,
 'split_algorithm': 'per-class-sha256-sort-v1',
 'ratios': {'train': 0.7, 'val': 0.15, 'test': 0.15},
 'counts': {'train': 5390, 'val': 1155, 'test': 1155},
 'pressure_shape': [240, 16, 16],
 'pressure_dtype': 'uint8',
 'labels': ['A',
  'B',
  'C',
  'D',
  'E',
  'F',
  'G',
  'H',
  'I',
  'J',
  'K',
  'L',
  'M',
  'N',
  'O',
  'P',
  'Q',
  'R',
  'S',
  'T',
  'U',
  'V',
  'W',
  'X',
  'Y',
  'Z',
  '1',
  '2',
  '3',
  '4',
  '5',
  '6',
  '7',
  '8',
  '9']}

In [6]:
# 6. 验证输出形状、类型和每类数量
expected_split_sizes = {
    name: count * len(LABELS)
    for name, count in SPLIT_COUNTS_PER_CLASS.items()
}

for split_name in ("train", "val", "test"):
    split_dir = OUTPUT_ROOT / split_name
    pressure = np.load(
        split_dir / "pressure.npy",
        mmap_mode="r",
        allow_pickle=False,
    )

    with (split_dir / "manifest.csv").open(
        "r",
        encoding="utf-8",
        newline="",
    ) as file:
        rows = list(csv.DictReader(file))

    class_counts = Counter(row["label"] for row in rows)
    expected_per_class = SPLIT_COUNTS_PER_CLASS[split_name]

    assert pressure.shape == (
        expected_split_sizes[split_name],
        *EXPECTED_SHAPE,
    )
    assert pressure.dtype == EXPECTED_DTYPE
    assert len(rows) == expected_split_sizes[split_name]
    assert set(class_counts) == set(LABELS)
    assert all(
        count == expected_per_class
        for count in class_counts.values()
    )

    print(
        f"{split_name}: shape={pressure.shape}, "
        f"dtype={pressure.dtype}, "
        f"每类={expected_per_class}"
    )

print("全部验证通过。")

train: shape=(5390, 240, 16, 16), dtype=uint8, 每类=154
val: shape=(1155, 240, 16, 16), dtype=uint8, 每类=33
test: shape=(1155, 240, 16, 16), dtype=uint8, 每类=33
全部验证通过。
